In [0]:
# TODOS
# 1. Centralize the write and read methods into a single notebook to be reused
# 2. Unit Test & Integration Test

import os
from pyspark.sql import functions as F
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, ShortType
)

# ── Paths ──────────────────────────────────────────────────────────────────

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
ALL_DATA_DIR = os.path.join(BASE_DIR, "data", "all_data_hhs")
BRONZE_PARQUET_DIR = os.path.join(BASE_DIR, "bronze_output", "parquet_data_hhs")

os.makedirs(ALL_DATA_DIR, exist_ok=True)
os.makedirs(BRONZE_PARQUET_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"ALL_DATA_DIR: {ALL_DATA_DIR}")
print(f"BRONZE_PARQUET_DIR: {BRONZE_PARQUET_DIR}")

In [0]:
# Write data to parquet
def write(input_df: DataFrame, out_dir):
    return input_df.write.mode('overwrite').parquet(out_dir)

In [0]:
# Reading parquet & create dataframe. 
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os
filepath_benefits = os.path.join(BRONZE_PARQUET_DIR, 'benefits')

filepath_rates = os.path.join(BRONZE_PARQUET_DIR, 'rates')

def read_parquet(filepath: str) -> DataFrame:
    data_f = spark.read.parquet(filepath)
    return data_f
    
df_benefits = read_parquet(filepath_benefits)
df_rates = read_parquet(filepath_rates)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def filter_baseline_rate(df: DataFrame) -> DataFrame:
    return (
        df
        .filter(
            (F.col("Age") == "30") &
            (F.col("Tobacco") == "No Preference")
        )
        .select(
            F.col("PlanId"),
            F.col("IndividualRate").cast("double").alias("IndividualRate")
        )
        .filter(F.col("IndividualRate").isNotNull())
        # .dropDuplicates(["PlanId"])
    )


df_rate_baseline = df_rates.transform(filter_baseline_rate)

display(df_rate_baseline)
print(f"Step 1 — baseline rate rows: {df_rate_baseline.count()}")

In [0]:
TARGET_SERVICE_PRIMARY_CARE = "Primary Care Visit to Treat an Injury or Illness"
TARGET_SERVICE_ROUTINE_DENTAL_ADULT = 'Routine Dental Services (Adult)'
TARGET_SERVICE_BASIC_DENTAL_ADULT = 'Basic Dental Care - Adult'
TARGET_SERVICE_EMERGENCY_TRANSPORT = 'Emergency Transportation/Ambulance'

# Step 2 — service benefit rows: 65704
# Step 2 — service benefit rows: 77353
# Step 2 — service benefit rows: 77353
# Step 2 — service benefit rows: 65704

from pyspark.sql import functions as F, DataFrame

def filter_service_benefits(data_df: DataFrame, targetService: str) -> DataFrame:
    return (
        data_df
        .filter(F.col('BenefitName') == targetService)
        .select(F.col('StandardComponentId').alias('PlanId'), 'BenefitName', 'CopayInnTier1')
        .filter(F.col("CopayInnTier1").isNotNull())
        .dropDuplicates(["PlanId"])
    )

# silver_layer_benefits = filter_service_benefits(df_benefits)
#    
df_benefits_primary_care = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_PRIMARY_CARE)
df_benefits_routine_dental = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_ROUTINE_DENTAL_ADULT)
df_benefits_basic_dental = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_BASIC_DENTAL_ADULT)
df_benefits_emergency_transport = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_EMERGENCY_TRANSPORT)

print(f"Step 2 — service benefit rows: {df_benefits_primary_care.count()}")
print(f"Step 2 — service benefit rows: {df_benefits_routine_dental.count()}")
print(f"Step 2 — service benefit rows: {df_benefits_basic_dental.count()}")
print(f"Step 2 — service benefit rows: {df_benefits_emergency_transport.count()}")


In [0]:
# DOLLAR_REGEX =  r"\$(\d+(?:\.\d+)?)"  # matches $30, $0, $15.50
# DOLLAR_REGEX = r"\d{1,3}(,\d{3})*(\.\d+)?" ##r"(?:\d+(?:\.\d+)?)?" 
DOLLAR_REGEX = r"\$([0-9,]+(?:\.[0-9]{2})?)" 


def parse_copay_amount(df: DataFrame) -> DataFrame:
    copay_lower = F.lower(F.col("CopayInnTier1"))

    is_no_charge   = copay_lower.rlike(r"no charge")
    is_coinsurance = copay_lower.rlike(r"coinsurance|%")
    has_dollar     = F.col("CopayInnTier1").rlike(DOLLAR_REGEX)
    #.rlike(r"\$\d")

    copay_dollar = (
        F.when(is_no_charge,   F.lit(0.0))
         .when(is_coinsurance, F.lit(None).cast("double"))  # exclude percentage-based
         .when(has_dollar,
               F.regexp_extract(F.col("CopayInnTier1"), DOLLAR_REGEX, 1).cast("double"))
         .otherwise(F.lit(None).cast("double"))             # 'Not Applicable', etc.
    )

    return (
        df
        .withColumn("copay_dollar", copay_dollar)
        .filter(F.col("copay_dollar").isNotNull())
        .select('PlanId', 'BenefitName', 'CopayInnTier1', 'copay_dollar')
    )

df_copay_primary_care = df_benefits_primary_care.transform(parse_copay_amount)
df_copay_routine_dental = df_benefits_routine_dental.transform(parse_copay_amount)
df_copay_basic_dental = df_benefits_basic_dental.transform(parse_copay_amount)
df_copay_emergency_transport = df_benefits_emergency_transport.transform(parse_copay_amount)


# display(df_copay_clean)
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_primary_care.count()}")
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_routine_dental.count()}")
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_basic_dental.count()}")
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_emergency_transport.count()}")

In [0]:
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, "silver", "data")
os.makedirs(SILVER_PARQUET_DIR, exist_ok=True)

write(df_rate_baseline, f"{SILVER_PARQUET_DIR}/rate_baseline")
write(df_copay_primary_care, f"{SILVER_PARQUET_DIR}/copay_primary_care")
write(df_copay_routine_dental, f"{SILVER_PARQUET_DIR}/copay_routine_dental")
write(df_copay_basic_dental, f"{SILVER_PARQUET_DIR}/copay_basic_dental")
write(df_copay_emergency_transport, f"{SILVER_PARQUET_DIR}/copay_emergency_transport")

In [0]:
# # Load Silver Layer data for Gold Layer processing
# filepath_rate_baseline = os.path.join(SILVER_PARQUET_DIR, 'rate_baseline')
# filepath_copay_primary_care = os.path.join(SILVER_PARQUET_DIR, 'copay_primary_care')
# filepath_copay_routine_dental = os.path.join(SILVER_PARQUET_DIR, 'copay_routine_dental')
# filepath_copay_basic_dental = os.path.join(SILVER_PARQUET_DIR, 'copay_basic_dental')
# filepath_copay_emergency_transport = os.path.join(SILVER_PARQUET_DIR, 'copay_emergency_transport')

# df_rate_baseline = read_parquet(filepath_rate_baseline)
# df_copay_primary_care = read_parquet(filepath_copay_primary_care)
# df_copay_routine_dental = read_parquet(filepath_copay_routine_dental)
# df_copay_basic_dental = read_parquet(filepath_copay_basic_dental)
# df_copay_emergency_transport = read_parquet(filepath_copay_emergency_transport)

In [0]:
# display(df_benefits.select('BenefitName').distinct())
# df_filtered_primary_care = df_benefits.filter(F.col('BenefitName').contains('Routine Dental Services (Adult)')).select('BenefitName')
# display(df_filtered_primary_care)